# What features do the models use?

**Short answer:** the OULAD and synthetic risk models use separate enrollment-level feature tables. They share four classifier families, but they use different datasets, fields, labels, and validation designs. They are not trained on a combined dataset.

This notebook lists the predictor columns after the model code removes labels, IDs, and audit-only fields. It also explains what each column means and when its source information is available.

> The synthetic track is simulated development data. Its feature list describes the current simulator and model code, not measurements from real students.


## At a glance

| Track | Model target | Predictors | Observation window | Evaluation split |
|---|---|---:|---|---|
| OULAD full-course baseline | Fail/Withdrawn = 1; Pass/Distinction = 0 | Up to 21 fields when all relevant tables are present | Whole course; includes unregistration date | Stratified random 75/25 enrollment holdout |
| OULAD early warning | Same OULAD target | Up to 20 fields at each cutoff (some optional by table availability) | 30, 60, or 90 course-relative days; no future unregistration date | Latest course presentation held out |
| Synthetic baseline | Simulated fail/withdrawal = 1; pass = 0 | 48 fields in the checked default generated snapshot | 30, 60, or 90 days after each offering starts | Student-disjoint train/validation/test |

“Up to” matters: the OULAD feature builder adds registration features only if the registration table is loaded, and VLE features only if both VLE tables are loaded. The inventory below describes the full expected OULAD schema.


## What counts as a model feature?

A **feature** is an input column the classifier can use to estimate enrollment risk. The outcome it is trying to predict is the **label**, not a feature. IDs are removed because they identify records; the model should learn patterns from the recorded context and behaviour.

The code uses four classifiers in each track: logistic regression, random forest, extra trees, and histogram gradient boosting. At each dataset and cutoff, they receive the same predictor columns. The numeric and categorical preprocessing is fitted within the training pipeline: numeric missing values use the training median, categorical missing values use the training mode, and categories are one-hot encoded. Logistic regression also standardizes numeric inputs; the tree models do not.


## OULAD: exact baseline predictors

The OULAD feature builder starts with `studentInfo`, creates the binary `target_risk`, then joins enrollment-level aggregates. The model removes `target_risk`, raw `final_result`, `id_student`, and `id_presentation` if present. **It retains `code_presentation` as an input** even though early-warning evaluation also holds out the latest presentation.

With the standard student information, assessments, registration, and VLE tables available, these are the 21 baseline predictors.


In [ ]:
import pandas as pd
from IPython.display import display

oulad_rows = [('Course context', 'code_module', 'Course/module code; categorical.', 'studentInfo'), ('Course context', 'code_presentation', 'Course presentation identifier (for example, a term offering); categorical and retained as a predictor.', 'studentInfo'), ('Student context', 'gender', 'Recorded gender category; included in this baseline, so fairness review is needed.', 'studentInfo'), ('Student context', 'region', 'Geographic region category.', 'studentInfo'), ('Student context', 'highest_education', 'Highest recorded prior education category.', 'studentInfo'), ('Student context', 'imd_band', 'Area-level deprivation band; included in this baseline, so fairness review is needed.', 'studentInfo'), ('Student context', 'age_band', 'Age band category.', 'studentInfo'), ('Student context', 'num_of_prev_attempts', 'Number of previous attempts at this module.', 'studentInfo'), ('Student context', 'studied_credits', 'Credits the student is registered to study.', 'studentInfo'), ('Student context', 'disability', 'Recorded disability category; included in this baseline, so fairness review is needed.', 'studentInfo'), ('Assessment progress', 'assessment_count', 'Number of distinct assessment IDs with a student assessment record.', 'studentAssessment + assessments'), ('Assessment progress', 'assessment_score_mean', 'Mean recorded assessment score; missing scores stay missing.', 'studentAssessment + assessments'), ('Assessment progress', 'assessment_score_std', 'Standard deviation of recorded assessment scores.', 'studentAssessment + assessments'), ('Assessment progress', 'assessment_score_min', 'Lowest recorded assessment score.', 'studentAssessment + assessments'), ('Assessment progress', 'assessment_score_max', 'Highest recorded assessment score.', 'studentAssessment + assessments'), ('Assessment progress', 'assessment_score_missing', 'Count of missing score values among the included assessment records.', 'studentAssessment + assessments'), ('Registration timing', 'date_registration', 'Mean course-relative registration date for the enrollment.', 'studentRegistration'), ('Registration timing', 'date_unregistration', 'Mean course-relative unregistration date; available to full-course baseline only.', 'studentRegistration'), ('Online activity', 'vle_clicks', 'Sum of recorded VLE clicks. Clicks are not study duration.', 'studentVle + vle'), ('Online activity', 'vle_active_days', 'Number of distinct course-relative days with VLE activity.', 'studentVle + vle'), ('Online activity', 'vle_resources', 'Number of distinct VLE resource IDs accessed.', 'studentVle + vle')]
oulad_inventory = pd.DataFrame(oulad_rows, columns=['Feature family','Feature column','Meaning','Source table(s)'])
display(oulad_inventory)
print(f'{len(oulad_inventory)} expected OULAD predictors when all feature source tables are available.')

### How OULAD features change at the early-warning cutoffs

The early-warning feature builder is run again for day 30, 60, and 90. It includes only assessment submissions with a non-missing `date_submitted` on or before the cutoff, and VLE records with a non-missing activity date on or before the cutoff. Thus assessment aggregates and VLE aggregates are recomputed from a shorter window. `date_unregistration` is removed because it could describe a future withdrawal. `date_registration` remains.

The other OULAD student and course fields stay in the table. This means the cutoff constrains the dated behaviour aggregates; it does not remove static demographic, enrollment, or course fields. The full-course and early-warning scores should not be treated as directly equivalent.


## Synthetic dataset: exact baseline predictors

The synthetic pipeline builds one feature row per student-course-offering enrollment. Events are filtered to the chosen cutoff before they are aggregated. Earlier-term academic history and prerequisite outcomes are used only when they predate the current course.

The current baseline removes record IDs, outcome labels, cutoff metadata, and audit-only student attributes. The feature inventory below was checked against the generator's default 1,000-student, three-term dataset and the predictor drop rules in the synthetic baseline code. It contains **48 predictor columns at each of days 30, 60, and 90**.


In [ ]:
synthetic_rows = [('Enrollment and course context', 'attempt_number', 'Attempt number for this student-course enrollment.'), ('Enrollment and course context', 'is_backlog', 'Whether this enrollment is a backlog/repeat.'), ('Enrollment and course context', 'enrollment_status', 'Status recorded on the enrollment row.'), ('Enrollment and course context', 'enrollment_date', 'Enrollment date.'), ('Enrollment and course context', 'term_id', 'Academic term identifier; retained as a categorical predictor.'), ('Enrollment and course context', 'section', 'Offering section code.'), ('Enrollment and course context', 'attendance_requirement_percent', 'Attendance threshold configured for the course offering.'), ('Enrollment and course context', 'term_number', 'Academic term sequence number.'), ('Enrollment and course context', 'course_type', 'Course type, such as theory or theory/lab.'), ('Enrollment and course context', 'credits', 'Course credit value.'), ('Enrollment and course context', 'difficulty', 'Configured course difficulty category.'), ('Enrollment and course context', 'estimated_effort_hours', 'Course-level estimated effort.'), ('Prior history and curriculum', 'prior_semester_gpa', 'Latest GPA from a completed earlier term; missing for first-term students.'), ('Prior history and curriculum', 'prior_cumulative_gpa', 'Latest cumulative GPA from a completed earlier term; missing for first-term students.'), ('Prior history and curriculum', 'prior_failed_credits', 'Failed credits in the latest completed earlier term.'), ('Prior history and curriculum', 'prior_backlog_count', 'Backlog count in the latest completed earlier term.'), ('Prior history and curriculum', 'prior_terms_completed', 'Count of earlier academic performance records.'), ('Prior history and curriculum', 'prerequisite_count', 'Number of curriculum prerequisites configured for the course.'), ('Prior history and curriculum', 'unmet_prerequisite_count', 'Prerequisites not passed before this course started.'), ('Attendance by cutoff', 'attendance_classes_observed', 'Count of attendance rows on or before the cutoff.'), ('Attendance by cutoff', 'attendance_absences', 'Count of absent attendance rows on or before the cutoff.'), ('Attendance by cutoff', 'attendance_pct_at_cutoff', 'Attendance percentage from the last included attendance row.'), ('Attendance by cutoff', 'attendance_pct_mean', 'Mean of the recorded running attendance percentages through cutoff.'), ('Assessment progress by cutoff', 'assessment_attempt_count', 'Count of included assessment attempt rows.'), ('Assessment progress by cutoff', 'assessment_mean_score', 'Mean marks obtained on assessments dated on or before cutoff.'), ('Assessment progress by cutoff', 'assessment_late_rate', 'Share of included attempts marked late.'), ('Assessment progress by cutoff', 'assessment_submitted_rate', 'Share of included attempts with submission_status equal to submitted.'), ('Study behaviour by cutoff', 'planned_sessions', 'Number of study sessions scheduled on or before cutoff.'), ('Study behaviour by cutoff', 'completed_sessions', 'Number of included sessions marked completed.'), ('Study behaviour by cutoff', 'active_study_minutes', 'Sum of active study minutes in included sessions.'), ('Study behaviour by cutoff', 'study_screen_minutes', 'Sum of screen-time minutes in included sessions.'), ('Study behaviour by cutoff', 'distraction_minutes', 'Sum of recorded distraction minutes in included sessions.'), ('Study behaviour by cutoff', 'actual_break_minutes', 'Sum of actual break minutes in included sessions.'), ('Study behaviour by cutoff', 'session_completion_rate', 'Completed sessions divided by planned sessions.'), ('Study behaviour by cutoff', 'mean_mastery_score', 'Mean observed topic mastery score by cutoff.'), ('Study behaviour by cutoff', 'mean_confidence_score', 'Mean observed topic confidence score by cutoff.'), ('Study behaviour by cutoff', 'mean_topic_fatigue', 'Mean observed topic fatigue score by cutoff.'), ('Study behaviour by cutoff', 'remaining_effort_hours', 'Sum of estimated effort remaining over observed topic states.'), ('Wellbeing and availability by cutoff', 'mean_total_screen_minutes', 'Mean daily total screen minutes through cutoff.'), ('Wellbeing and availability by cutoff', 'mean_total_break_minutes', 'Mean daily total break minutes through cutoff.'), ('Wellbeing and availability by cutoff', 'mean_sleep_hours', 'Mean daily sleep hours through cutoff.'), ('Wellbeing and availability by cutoff', 'mean_sleep_deficit_hours', 'Mean daily sleep deficit hours through cutoff.'), ('Wellbeing and availability by cutoff', 'mean_fatigue_score', 'Mean daily fatigue score through cutoff.'), ('Wellbeing and availability by cutoff', 'mean_concentration_score', 'Mean daily concentration score through cutoff.'), ('Wellbeing and availability by cutoff', 'mean_stress_score', 'Mean daily stress score through cutoff.'), ('Wellbeing and availability by cutoff', 'mean_available_hours', 'Mean daily available study hours through cutoff.'), ('Wellbeing and availability by cutoff', 'mean_commute_hours', 'Mean daily commute hours through cutoff.'), ('Wellbeing and availability by cutoff', 'mean_part_time_work_hours', 'Mean daily part-time work hours through cutoff.')]
synthetic_inventory = pd.DataFrame(synthetic_rows, columns=['Feature family','Feature column','Meaning'])
family_counts = (synthetic_inventory.groupby('Feature family', sort=False).size()
                 .rename('Predictor count').reset_index())
display(family_counts)
display(synthetic_inventory)
print(f'{len(synthetic_inventory)} synthetic predictors at each evaluated cutoff in the checked snapshot.')

### Check against GitHub's actual training code

I traced the synthetic input matrix through the repository call chain: the synthetic baseline notebook calls `run_synthetic_baselines`; that function builds cutoff features and removes the model module's `DROP_COLUMNS` before fitting. That drop set includes the `AUDIT_ONLY_COLUMNS` set.

There is a mismatch in the current GitHub materials: the feature-inventory cell in Notebook 07 omits those audit-only columns from its example exclusion list, and the formal baseline report says 55 predictors while including some fields the model code drops. The training code is the source of truth for what the classifiers actually receive. Applying its drop set to the checked default synthetic feature table gives **48 predictor columns** at each cutoff. The inventory above reflects those 48.


In [ ]:
# Optional live check against the repository modules and generated CSVs.
# This works from a checkout containing the synthetic source modules and data/raw/synthetic/.
from pathlib import Path

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / 'src').exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
SYNTHETIC_DIR = PROJECT_ROOT / 'data' / 'raw' / 'synthetic'

try:
    from src.features.synthetic_features import build_cutoff_features
    from src.models.synthetic_baseline import DROP_COLUMNS
except ImportError as exc:
    print(f'Live check unavailable in this checkout: {exc}')
else:
    if not SYNTHETIC_DIR.exists():
        print(f'Generate the synthetic CSVs first; expected directory: {SYNTHETIC_DIR}')
    else:
        actual_features = build_cutoff_features(SYNTHETIC_DIR, (30, 60, 90))
        expected = set(synthetic_inventory['Feature column'])
        for cutoff, frame in actual_features.items():
            actual = set(frame.columns) - set(DROP_COLUMNS)
            print(f'Day {cutoff}: {len(actual)} model predictors; matches this inventory: {actual == expected}')
            if actual != expected:
                print('Only in generated matrix:', sorted(actual - expected))
                print('Only in notebook inventory:', sorted(expected - actual))


### Synthetic fields deliberately kept out of the model

- **Identifiers:** `enrollment_id`, `student_id`, `offering_id`, `course_id`, `faculty_id`, `department_id`, `program_id`.
- **Labels/outcomes:** `risk_label`, `passed`, `withdrawn`, `requires_retake`. These define the prediction target or encode the final outcome.
- **Cutoff metadata:** `observation_cutoff_days`; `cutoff_date` and `start_date` are dropped while building the feature table.
- **Audit-only attributes:** `category`, `consent_status`, `preferred_language`, `residence_type`, `employment_status`, `study_mode`, `prior_education_stream`, `first_generation_student`, `internet_access_type`, and `financial_aid_status`. These can be used for subgroup review, but are not predictors in the default synthetic baseline.
- **Future/current final academic performance:** final outcomes and current-term academic-performance summaries are not predictor inputs.

Some demographic and access attributes are present in the generated `students` table but are excluded from the synthetic predictors by the baseline model's explicit audit-only drop set. The OULAD baseline has a different policy: it currently includes several demographic attributes, including `gender`, `imd_band`, and `disability`.


## Feature families side by side

| What the model sees | OULAD | Synthetic |
|---|---|---|
| Course/enrollment context | Module and presentation codes, prior attempts, studied credits | Attempt/backlog status, term/section, course type, credits, difficulty, effort estimate, attendance requirement |
| Academic history | Assessment score aggregates over available assessment records | Prior completed-term GPA/backlogs, prerequisites, cutoff-bounded assessment attempts |
| Participation | VLE click count, active days, resource count | Attendance rollups and completed/planned study-session behaviour |
| Time and wellbeing | Registration/unregistration timing; no sleep or daily availability features in this OULAD model | Daily sleep, stress, fatigue, concentration, screen/break time, availability, commute, work hours |
| Protected/sensitive fields | Some demographic fields are model predictors | Sensitive and consent-related fields are excluded from predictors and reserved for audit |

The synthetic track has more planner-specific measures because its simulator creates them. That wider feature set does **not** mean those measurements or relationships have been validated in real students.


## Important interpretation notes

1. **Same algorithms does not mean same model.** Each dataset has its own feature builder, target, preprocessing fit, split, and model parameters. A score from one track should not be ranked against a score from the other as if they came from the same population and task.
2. **The OULAD full-course baseline is retrospective.** It uses course-wide assessment/activity summaries and unregistration timing. Early warning removes future unregistration and time-filters dated events, but the label remains the final result.
3. **The synthetic benchmark tests the pipeline under designed conditions.** Its simulated latent student profiles intentionally produce correlated records. This checks software behaviour; it does not establish real-world predictive validity.
4. **Feature inclusion is not feature importance.** This notebook says what the models are allowed to use. It does not say which fields the fitted models rely on most, or whether any field causes an outcome.
5. **Fairness needs separate review.** OULAD includes some demographic predictors. Synthetic audit-only fields are excluded from its default predictor matrix, but subgroup performance should still be checked.


## Code trail

- OULAD feature construction: `src/features/feature_engineering.py`
- OULAD predictor exclusions and preprocessing: `src/models/baseline.py`
- OULAD baseline notebook: `notebooks/03_oulad_baseline_comparison.ipynb`
- OULAD cutoff evaluation: `notebooks/04_oulad_early_warning_evaluation.ipynb`
- Synthetic feature construction: `src/features/synthetic_features.py`
- Synthetic predictor exclusions and preprocessing: `src/models/synthetic_baseline.py`
- Synthetic feature and model notebooks: `notebooks/07_synthetic_processing_and_features.ipynb`, `notebooks/08_synthetic_baseline_testing.ipynb`

The exact synthetic source files reviewed are available at the repository snapshot `f589a8f50b4c580669755b246bc0b1788120df7b`: [synthetic feature builder](https://github.com/Rafi-Sojan/Smart-Study-Planner/blob/f589a8f50b4c580669755b246bc0b1788120df7b/src/features/synthetic_features.py), [synthetic model input and exclusions](https://github.com/Rafi-Sojan/Smart-Study-Planner/blob/f589a8f50b4c580669755b246bc0b1788120df7b/src/models/synthetic_baseline.py), [Notebook 07](https://github.com/Rafi-Sojan/Smart-Study-Planner/blob/f589a8f50b4c580669755b246bc0b1788120df7b/notebooks/07_synthetic_processing_and_features.ipynb), and [Notebook 08](https://github.com/Rafi-Sojan/Smart-Study-Planner/blob/f589a8f50b4c580669755b246bc0b1788120df7b/notebooks/08_synthetic_baseline_testing.ipynb).

The OULAD and synthetic pipelines are intentionally separate. Keep the two training tables separate as well.
